# PACE MiniGrid full training and evaluation on Kaggle

Use **GPU**, **Internet** and **Save & Run All** for training and checkpoint evaluation. Leave `SOURCE` empty to clone the pinned trainer, or set it to an attached source ZIP or folder.

1. **Session 1 (default)** starts `algorithm1` on seed 0, with a budget of 30,000 PPO batches and explicit provisional alpha 0.01. It pauses at main-loop batch 21,000.
2. **Session 2** attaches the previous notebook output with **Add Input** and reads `RESUME_CHECKPOINT`. Both training sessions save a checkpoint every 250 main-loop batches and one row per batch in `learning_curve.csv`. If either session reaches the 10-hour training limit, continue from the checkpoint in `run_summary.json` using Session 2.
3. **Session 3** attaches both training outputs and reads the run folders from `ANALYSIS_RUN_DIRS`. It combines training curves, evaluates OOD around every 500 batches with 20 episodes per layout (100 at the final checkpoint), and saves repeated scores for CV at up to five checkpoints. With `OOD_LAYOUTS_JSON` empty the suite uses seven reference layouts. Supply the five missing verified grids for the 12-layout paper suite.
4. **Session 4** reads completed `final_ood.json` files listed in `FINAL_OOD_REPORTS` and exports data for the bar chart, IQM and Optimality Gap. Confidence intervals use multiple seeds. This session can run on CPU.


In [ ]:
import os
import shutil
import subprocess
import sys
import tempfile
import zipfile
from pathlib import Path
from time import perf_counter
from textwrap import dedent

source_start = perf_counter()

SESSION = 1
SEED = 0
PROVISIONAL_ALPHA = 0.01  # Explicit reproduction choice.
RESUME_CHECKPOINT = ''  # Session 2: paste the attached .msgpack path.
ANALYSIS_RUN_DIRS = []  # Session 3: run folders that contain the checkpoints/ directory.
OOD_LAYOUTS_JSON = ''  # Optional paper suite asset file with the five missing grids.
OOD_EVERY = 500
OOD_CURVE_EPISODES = 20
OOD_FINAL_EPISODES = 100
CV_LEVELS = ['16Rooms', 'Labyrinth', 'Labyrinth2', 'Maze', 'Maze2']
CV_LAYOUTS_JSON = ''  # Optional custom grids for all names in CV_LEVELS.
FINAL_OOD_REPORTS = []  # Session 4: final_ood.json paths from completed seeds.
PAUSE_AFTER_BATCH = 21_000
CHECKPOINT_INTERVAL = 250
TARGET_PPO_BATCHES = 30_000
SESSION_LIMIT_SECONDS = 10 * 60 * 60

if SESSION not in (1, 2, 3, 4):
    raise ValueError('Set SESSION to 1, 2, 3, or 4.')
checkpoint_input = Path(RESUME_CHECKPOINT).resolve() if RESUME_CHECKPOINT else None
if SESSION == 2 and (checkpoint_input is None or not checkpoint_input.is_file()):
    raise FileNotFoundError('Attach the previous output and set RESUME_CHECKPOINT to its .msgpack file.')
if SESSION == 1 and checkpoint_input is not None:
    raise ValueError('Set SESSION to 2 to use an existing checkpoint.')

WORK = Path('/kaggle/working')
runs = WORK / 'pace'
runs.mkdir(parents=True, exist_ok=True)
RUN = Path(tempfile.mkdtemp(prefix=f'algorithm1-seed{SEED}-session{SESSION}-', dir=runs))
print(f'Session output: {RUN}', flush=True)
REPO_URL = 'https://github.com/SimforianusJonathan/PACE-Research.git'
REPO_REF = '5340ffc6ddf9bb8a7c495af689e03e5ebd6a9382'
SOURCE = ''  # Optional path to a source ZIP or directory under /kaggle/input/.
source_id = f'local-{Path(SOURCE).stem}' if SOURCE else REPO_REF.replace('/', '-')[:12]
REPO = WORK / f'PACE-Research-{source_id}'
RUNTIME = Path('/tmp/pace-kaggle')

if not REPO.exists():
    if SOURCE:
        source = Path(SOURCE)
        if source.is_dir():
            shutil.copytree(source, REPO)
        else:
            with zipfile.ZipFile(source) as archive:
                archive.extractall(REPO)
    else:
        subprocess.run(['git', 'clone', REPO_URL, str(REPO)], check=True)
        subprocess.run(['git', 'checkout', '--detach', REPO_REF], cwd=REPO, check=True)

if not (REPO / 'scripts/setup_kaggle.sh').is_file():
    raise RuntimeError('This source lacks Kaggle support. Use a Git ref or source archive containing this notebook.')
if not (REPO / 'train/pace_analysis.py').is_file():
    raise RuntimeError('Use the pinned Git revision or an attached source with train/pace_analysis.py.')
if (REPO / '.git').exists():
    subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=REPO, check=True)

if SESSION == 4:
    env = {**os.environ, 'PACE_RUNTIME_DIR': str(RUNTIME), 'JAX_PLATFORMS': 'cpu',
           'MPLBACKEND': 'Agg', 'PYTHONUNBUFFERED': '1'}
else:
    # Use Kaggle's unmodified driver environment for this diagnostic.
    smi = shutil.which('nvidia-smi')
    if not smi or subprocess.run([smi, '-L'], check=False).returncode != 0:
        print('nvidia-smi did not list a GPU. Checking the CUDA driver directly.')

    visible_device = os.environ.get('CUDA_VISIBLE_DEVICES', '0').split(',', 1)[0]
    env = {**os.environ, 'PACE_RUNTIME_DIR': str(RUNTIME), 'CUDA_VISIBLE_DEVICES': visible_device,
           'JAX_PLATFORMS': 'cuda', 'XLA_PYTHON_CLIENT_PREALLOCATE': 'false',
           'MPLBACKEND': 'Agg', 'PYTHONUNBUFFERED': '1'}
    # Keep mounted driver libraries while letting the wheels provide the CUDA toolkit.
    paths = ['/usr/local/nvidia/lib64', *os.environ.get('LD_LIBRARY_PATH', '').split(os.pathsep)]
    driver_paths = dict.fromkeys(path for path in paths if Path(path).is_absolute()
                                and Path(path).name not in {'compat', 'stubs'}
                                and (Path(path) / 'libcuda.so.1').is_file())
    if driver_paths:
        env['LD_LIBRARY_PATH'] = os.pathsep.join(driver_paths)
    else:
        env.pop('LD_LIBRARY_PATH', None)

    driver_probe = dedent('''
    import ctypes
    cuda = ctypes.CDLL('libcuda.so.1')
    status = cuda.cuInit(0)
    if status != 0:
        raise RuntimeError(f'CUDA driver initialization failed (cuInit={status})')
    count = ctypes.c_int()
    status = cuda.cuDeviceGetCount(ctypes.byref(count))
    if status != 0 or count.value < 1:
        raise RuntimeError(f'No visible CUDA device (status={status}, count={count.value})')
    ''')
    try:
        subprocess.run([sys.executable, '-c', driver_probe], env=env, check=True)
    except subprocess.CalledProcessError:
        raise RuntimeError('Kaggle has no usable CUDA device for this run. Select a GPU accelerator, '
                           'then restart the session or save a new GPU-enabled version.') from None

source_seconds = perf_counter() - source_start

In [ ]:
from threading import Event, Timer

def run_logged(command, log_path, timeout_seconds=None):
    log_path.parent.mkdir(parents=True, exist_ok=True)
    print(f'Log: {log_path}', flush=True)
    with log_path.open('w', encoding='utf-8') as log, subprocess.Popen(
        command, cwd=REPO, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, encoding='utf-8', errors='replace',
    ) as process:
        stopped_for_time = Event()

        def stop_training():
            if process.poll() is None:
                stopped_for_time.set()
                process.terminate()

        timer = None
        if timeout_seconds is not None:
            timer = Timer(timeout_seconds, stop_training)
            timer.daemon = True
            timer.start()
        try:
            for line in process.stdout:
                log.write(line)
                log.flush()
                print(line, end='', flush=True)
            code = process.wait()
        finally:
            if timer is not None:
                timer.cancel()
                timer.join()

    if code and not stopped_for_time.is_set():
        raise RuntimeError(f'{log_path.stem} failed with exit code {code}. Full log: {log_path}')
    return stopped_for_time.is_set()

setup_start = perf_counter()
run_logged(['bash', 'scripts/setup_kaggle.sh'], RUN / 'setup.log')
PACE_PYTHON = str(RUNTIME / 'venv/bin/python')

probe = '''
import importlib.metadata as metadata
import json
import jax
import jax.numpy as jnp
from jaxued.environments import Maze
from train.pace_minigrid import MiniGridConfig
config = MiniGridConfig(provisional_alpha=0.01, protocol='algorithm1')
if not (config.uses_prefill and config.separate_replay_rollouts):
    raise RuntimeError('Use a trainer revision with scored prefill and separate replay rollouts.')
devices = jax.local_devices(backend='gpu')
assert len(devices) == 1, devices
result = jax.jit(lambda x: x @ x)(jnp.ones((16, 16))).block_until_ready()
assert result.device.platform == 'gpu' and float(result[0, 0]) == 16.0
pin = '0f8f1284677375b889e4f13a32c9617cd009f8c4'
origin = json.loads(metadata.distribution('jaxued').read_text('direct_url.json'))
assert origin['vcs_info']['commit_id'] == pin
print({'device': str(devices[0]), 'jax': jax.__version__, 'jaxued_commit': pin, 'environment': Maze.__name__})
'''
if SESSION != 4:
    run_logged([PACE_PYTHON, '-c', probe], RUN / 'gpu-probe.log')
setup_seconds = source_seconds + perf_counter() - setup_start

In [ ]:
if SESSION in (1, 2):
    import json

    expected_config = {
        'seed': SEED,
        'protocol': 'algorithm1',
        'budget': 'ppo',
        'num_updates': TARGET_PPO_BATCHES,
        'provisional_alpha': PROVISIONAL_ALPHA,
    }
    base = [PACE_PYTHON, '-m', 'train.pace_minigrid']
    common = ['--device', 'gpu', '--output-dir', str(RUN),
              '--checkpoint-interval', str(CHECKPOINT_INTERVAL), '--log-learning-curve']

    if SESSION == 1:
        command = base + [
            'train', '--seed', str(SEED), '--protocol', 'algorithm1', '--budget', 'ppo',
            '--provisional-alpha', str(PROVISIONAL_ALPHA), '--stop-after', str(PAUSE_AFTER_BATCH),
        ] + common
    else:
        checkpoint_check = dedent('''
    import json
    import sys
    from pathlib import Path
    from train.pace_checkpoint import read_checkpoint

    stored = read_checkpoint(Path(sys.argv[1]))['config']
    expected = json.loads(sys.argv[2])
    differences = {name: {'checkpoint': stored.get(name), 'expected': value}
                   for name, value in expected.items() if stored.get(name) != value}
    if differences:
        raise ValueError(f'Checkpoint configuration mismatch: {differences}')
    ''')
        run_logged([PACE_PYTHON, '-c', checkpoint_check, str(checkpoint_input),
                    json.dumps(expected_config)], RUN / 'checkpoint-check.log')
        command = base + ['resume', '--checkpoint', str(checkpoint_input)] + common

    train_log = RUN / 'train.log'
    start = perf_counter()
    timed_out = run_logged(command, train_log, timeout_seconds=SESSION_LIMIT_SECONDS)
    training_seconds = perf_counter() - start

In [ ]:
if SESSION in (1, 2):
    records = []
    for line in train_log.read_text(encoding='utf-8').splitlines():
        if line.startswith('{'):
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError:
                if not timed_out:
                    raise

    start_record = next((record for record in records if 'config' in record), None)
    last_checkpoint = next(
        (record for record in reversed(records)
         if record.get('checkpoint') and Path(record['checkpoint']).is_file()),
        None,
    )
    if start_record is None or last_checkpoint is None:
        raise RuntimeError(f'Training ended without a logged checkpoint. See {train_log}')

    actual_config = start_record['config']
    if any(actual_config.get(name) != value for name, value in expected_config.items()):
        raise RuntimeError(f'Unexpected training configuration: {actual_config}')
    if last_checkpoint['ppo_batches'] == TARGET_PPO_BATCHES:
        status = 'training_complete'
    elif timed_out:
        status = 'session_time_limit'
    elif SESSION == 1 and last_checkpoint['rollout_batches'] == PAUSE_AFTER_BATCH:
        status = 'session_1_complete'
    else:
        raise RuntimeError(f'Unexpected checkpoint boundary: {last_checkpoint}')

    seconds = round(training_seconds, 2)
    # Fresh training starts at zero even though the first config log follows prefill.
    incoming_env_steps = start_record['env_steps'] if SESSION == 2 else 0
    actual_new_steps = last_checkpoint['env_steps'] - incoming_env_steps
    summary = {
        'status': status,
        'session': SESSION,
        'trainer_ref': REPO_REF if not SOURCE else None,
        'input_checkpoint': str(checkpoint_input) if checkpoint_input else None,
        'checkpoint': last_checkpoint['checkpoint'],
        'config': actual_config,
        'counts': last_checkpoint,
        'learning_curve': str(RUN / 'learning_curve.csv'),
        'session_env_steps_saved': actual_new_steps,
        'timing': {
            'setup_seconds': round(setup_seconds, 2),
            'training_seconds': seconds,
        },
    }
    report_path = RUN / 'run_summary.json'
    report_path.write_text(json.dumps(summary, indent=2) + '\n', encoding='utf-8')
    print(json.dumps(summary, indent=2))
    print(f'Session report: {report_path}')

In [ ]:
import json

if SESSION == 3:
    if not ANALYSIS_RUN_DIRS:
        raise ValueError('Attach training outputs and fill ANALYSIS_RUN_DIRS with the run folders.')
    suite = 'paper' if OOD_LAYOUTS_JSON else 'reference'
    command = [
        PACE_PYTHON, '-m', 'train.pace_analysis', 'evaluate', '--output-dir', str(RUN),
        '--suite', suite, '--ood-every', str(OOD_EVERY),
        '--curve-episodes', str(OOD_CURVE_EPISODES), '--final-episodes', str(OOD_FINAL_EPISODES),
        '--cv-levels', *CV_LEVELS, '--run-dirs', *ANALYSIS_RUN_DIRS,
    ]
    if OOD_LAYOUTS_JSON:
        command += ['--levels-json', OOD_LAYOUTS_JSON]
    if CV_LAYOUTS_JSON:
        command += ['--cv-levels-json', CV_LAYOUTS_JSON]
    run_logged(command, RUN / 'analysis.log')
    report = RUN / 'final_ood.json'
    summary = json.loads(report.read_text(encoding='utf-8'))
    print(json.dumps(summary, indent=2))
    if summary['training_complete']:
        run_logged([PACE_PYTHON, '-m', 'train.pace_analysis', 'aggregate',
                    '--reports', str(report), '--output-dir', str(RUN)], RUN / 'aggregate.log')

if SESSION == 4:
    if not FINAL_OOD_REPORTS:
        raise ValueError('Attach the evaluation outputs and fill FINAL_OOD_REPORTS.')
    run_logged([PACE_PYTHON, '-m', 'train.pace_analysis', 'aggregate',
                '--reports', *FINAL_OOD_REPORTS, '--output-dir', str(RUN)], RUN / 'aggregate.log')
    print((RUN / 'ood_aggregate.json').read_text(encoding='utf-8'))
